# SatQuery AI — Module M1: EarthDial GPU Inference Server
This notebook runs the official **EarthDial 4B RGB VLM** on a free **Google Colab T4 GPU (16 GB VRAM)** and exposes an inference endpoint for the local **SatQuery AI M1/M4 modules**.

### Step 0: Ensure GPU is Enabled
Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.

In [ ]:
# Step 1: Verify CUDA GPU
!nvidia-smi

In [ ]:
# Step 2: Install required packages for EarthDial and FastAPI
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.37.2 tokenizers==0.15.1 accelerate peft bitsandbytes timm einops pillow pydantic fastapi uvicorn nest-asyncio pyngrok python-multipart

In [ ]:
# Step 3: Load EarthDial Model into GPU Memory (BF16)
import torch
from transformers import AutoTokenizer, AutoModel
from torchvision import transforms
from PIL import Image
import io, base64

MODEL_ID = "akshaydudhane/EarthDial_4B_RGB"
print(f"[*] Downloading & loading model: {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=False)
model = AutoModel.from_pretrained(
    MODEL_ID,
    low_cpu_mem_usage=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
).eval()

image_size = getattr(model.config, "force_image_size", None) or \
             getattr(model.config.vision_config, "image_size", 448)

transform = transforms.Compose([
    transforms.Resize((image_size, image_size), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
print("[+] EarthDial model successfully loaded on T4 GPU!")

In [ ]:
# Step 4: Define FastAPI App
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="SatQuery EarthDial API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class AnalyzeRequest(BaseModel):
    image_base64: str
    question: str
    num_beams: int = 5
    temperature: float = 0.0
    max_new_tokens: int = 128

@app.get("/health")
def health():
    return {"status": "ready", "model": MODEL_ID, "cuda": torch.cuda.is_available()}

@app.post("/analyze")
def analyze(req: AnalyzeRequest):
    try:
        img_bytes = base64.b64decode(req.image_base64)
        img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Invalid image: {e}")

    pixel_values = transform(img).unsqueeze(0).cuda().to(torch.bfloat16)
    generation_config = {
        "num_beams": req.num_beams,
        "max_new_tokens": req.max_new_tokens,
        "min_new_tokens": 1,
        "do_sample": req.temperature > 0.0,
        "temperature": req.temperature if req.temperature > 0.0 else 1.0,
    }
    with torch.no_grad():
        answer = model.chat(
            tokenizer=tokenizer,
            pixel_values=pixel_values,
            question=req.question,
            generation_config=generation_config,
            verbose=False
        )
    return {"answer": str(answer).strip(), "model": MODEL_ID}

In [ ]:
# Step 5: Start Server with Public Tunnel (localtunnel or cloudflared)
# Expose using cloudflared
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

import subprocess, time
# Start cloudflared tunnel in background on port 8000
tunnel_proc = subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8000"], stderr=subprocess.PIPE, text=True)
time.sleep(4)

# Read tunnel URL from logs
print("[*] Finding public tunnel URL...")
public_url = None
for _ in range(15):
    line = tunnel_proc.stderr.readline()
    if "trycloudflare.com" in line:
        for part in line.split():
            if "trycloudflare.com" in part:
                public_url = part.strip()
                break
    if public_url:
        break
    time.sleep(1)

if public_url:
    print(f"\n=======================================================")
    print(f"🚀 EARTHDIAL GPU SERVER IS LIVE!")
    print(f"Set this environment variable in your local SatQuery-AI project:")
    print(f"$env:EARTHDIAL_API_URL='{public_url}'")
    print(f"=======================================================\n")
else:
    print("Tunnel started. Check log output for trycloudflare.com URL.")

# Run Uvicorn server (blocks this cell while serving)
import uvicorn
uvicorn.run(app, host="0.0.0.0", port=8000)